In [1]:
import os
import json
import tempfile
import subprocess
from itertools import islice

import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download


/Users/aneeshkalla/anaconda3/envs/driving-tool/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/aneeshkalla/anaconda3/envs/driving-tool/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Creating nexar dataframe
# Now that we know the CSV structure, we can transform it according to the user’s requested format.
import json

src_path = "nexar_train.csv"
out_path = "nexar_train_prepared.h5"

df = pd.read_csv(src_path)

# Build the target dataframe
final_df = pd.DataFrame()
final_df["filepath"] = df["filepath"]
final_df["source"] = "huggingface"
final_df["event_time"] = df["time_of_event"]

# Infer collision vs no collision based on presence of time_of_event
final_df["event_description"] = df["time_of_event"].apply(
    lambda x: "collision" if pd.notna(x) else "no collision"
)

final_df["split"] = df["split"] if "split" in df.columns else "train"

# Bundle all other columns into `other_info`
used_cols = {"filepath", "time_of_event", "split"}
other_info_cols = [c for c in df.columns if c not in used_cols]

final_df["other_info"] = df[other_info_cols].to_dict(orient="records")
final_df["other_info"] = final_df["other_info"].apply(json.dumps)

# Save to .h5
final_df.to_hdf(out_path, key="data", mode="w")
out_path = "nexar_train_prepared.h5"

df = pd.read_hdf(out_path, key="data")

df


,filepath,source,event_time,event_description,split,other_info
0,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.500,collision,train,"{""time_of_alert"": 18.633, ""light_conditions"": ..."
1,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.800,collision,train,"{""time_of_alert"": 19.233, ""light_conditions"": ..."
2,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,20.101,collision,train,"{""time_of_alert"": 20.068, ""light_conditions"": ..."
3,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.267,collision,train,"{""time_of_alert"": 18.333, ""light_conditions"": ..."
4,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,21.146,collision,train,"{""time_of_alert"": 18.196, ""light_conditions"": ..."
...,...,...,...,...,...,...
1495,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1496,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1497,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1498,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."


In [ ]:
out_path = "nexar_train_prepared.h5"

df = pd.read_hdf(out_path, key="data")

df